## 1. Import packages and set options 
<a name="import_packages"></a>

In [ ]:
import pandas as pd  # a module which provides the data structures and functions to store and manipulate tables in dataframes
import pydbtools as pydb  # A module which allows SQL queries to be run on the Analytical Platform from Python, see https://github.com/moj-analytical-services/pydbtools
import boto3  # allows you to directly create, update, and delete AWS resources from Python scripts
import numpy as np
import re

#import the pacakges required to produce an excel spreadsheet
import boto3, io, os, pathlib

# sets parameters to view dataframes for tables easier
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

## 2. Define key variables to be used throughout the notebook 
<a name="define_key_variables"></a>

In [ ]:
#this is the database we will be extracting from
database = "familyman_live_v4"

#this extracts the latest snapshot from athena
get_snapshot_date = f"SELECT mojap_snapshot_date from {database}.events order by mojap_snapshot_date desc limit 1"
snapshot_date = str(pydb.read_sql_query(get_snapshot_date)['mojap_snapshot_date'].values[0])

#this extracts the February snapshot from athena
snapshot_date = "2023-02-09"

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#setting current year and quarter
pub_quarter = 4
pub_year = 2022

#### Create a table with cases in which care order applications were made

In [ ]:
pydb.create_temp_table (
f"""
SELECT
  DISTINCT
  case_number
FROM
  {fcsq_database}.ca_apps_order_types
WHERE
  order_code = 1
  
""",
"care_order_apps")

#### Create a table with cases in which interim care orders were granted

In [ ]:
pydb.create_temp_table (
f"""
SELECT
  DISTINCT
  case_number
FROM
  {fcsq_database}.ca_all_disposals
WHERE
  order_code = 40
  
""",
"interim_care_orders")

In [ ]:
pydb.create_temp_table (
f"""
SELECT
  year,
  case_number,
  CASE WHEN case_number IN (SELECT case_number FROM __temp__.care_order_apps)
                              THEN 1 ELSE 0 END
        AS care_app,
  CASE WHEN case_number IN (SELECT case_number FROM __temp__.interim_care_orders)
                              THEN 1 ELSE 0 END
        AS interim_care_ord
FROM
  {fcsq_database}.ca_closed_cases
WHERE 
  case_type = 'C'
  AND year BETWEEN 2011 and 2021 
  
""",
    
"care_case_flags")

In [ ]:
pydb.read_sql_query ("select * from __temp__.care_case_flags")

In [ ]:
pydb.read_sql_query ("select year, count(*) as cases, sum(care_app) as care_apps, sum(interim_care_ord) as interim_care_ords from __temp__.care_case_flags group by year order by year")